# Revisão humana, aprovação e rollback

POC DEMO — contrato PROVISÓRIO / PENDENTE DE HOMOLOGAÇÃO. Execute em kernel novo. Credenciais somente por ambiente.


Configuração tipada. Pré-requisitos: Docker saudável, seed DEMO, token por ambiente. Entradas: parâmetros abaixo; saídas: tabelas, IDs e relatório JSON/Markdown. Escritas exigem confirmação EXECUTE_DEMO e chave estável; repetir chave igual recupera o mesmo job. Espere READY; erros de dependência exigem verificar docker compose logs.

In [1]:
import os, json
import pandas as pd
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
cfg=Settings.load()
from src.workbench.notebook_config import Parameters
from src.workbench.operations_sdk import call, report, await_job
params=Parameters.load()
display(params.model_dump(mode="json"))
display({"state":"READY", "write_enabled":params.execute_write, "tracking":"remote Docker service", "browser_ui":cfg.ui})
evidence={}
NOTEBOOK_ID="07_registry_governanca"


{'mode': 'READ_EXISTING',
 'proposal_id': '712b11cd-a96a-4bb9-83bb-691724ceae88',
 'snapshot_id': None,
 'model_name': 'planilha-turmas-demo',
 'model_version': '',
 'registry_prefix': '',
 'registry_page_token': '',
 'campaign_id': None,
 'approval_id': None,
 'execute_write': False,
 'confirmation': '',
 'request_key': '',
 'requester': 'scientist-demo',
 'algorithm': 'ridge',
 'alpha': 1.0,
 'capacity': 40,
 'model_count': 30,
 'parallel_jobs': 2,
 'campaign_action': 'observe',
 'governance_action': 'read',
 'review_action': 'APPROVE',
 'reason': '',
 'lineage_kind': 'proposal',
 'external_id': '',
 'reset_mode': 'preview',
 'reinitialize': 'demo-ready',
 'reset_plan_id': '8a186e1f-3eb9-4819-b6ec-f99ff331853e'}

{'state': 'READY',
 'write_enabled': False,
 'tracking': 'remote Docker service',
 'browser_ui': 'http://127.0.0.1:5000'}

1. Modelos/versões e pedidos de revisão. Registro é candidato; aprovação exige token separado de operador e versão explícita.

In [2]:
import mlflow
mlflow.set_tracking_uri(cfg.tracking);client=mlflow.MlflowClient()
from urllib.parse import urlencode
catalog=call(cfg,"GET","/v1/governance/models?"+urlencode(dict(prefix=params.registry_prefix,page_token=params.registry_page_token,limit=100)))
display(pd.DataFrame(catalog["items"]));display({"next_page_token":catalog["next_page_token"]})
versions=client.search_model_versions("name='"+params.model_name+"'",max_results=100)
display(pd.DataFrame([dict(model=v.name,version=v.version,run_id=v.run_id,tags=v.tags) for v in versions]))
display(pd.DataFrame(call(cfg,"GET","/v1/governance/requests")))


,model_name,aliases,tags
0,C1-CR1-P1,{},{}
1,C1-CR1-P2,{},{}
2,C1-CR2-P1,{},{}
3,C1-CR2-P2,{},{}
4,C1-CR3-P1,{},{}
5,C1-CR3-P2,{},{}
6,C1-CR4-P1,{},{}
7,C1-CR4-P2,{},{}
8,C1-CR5-P1,{},{}
9,C1-CR5-P2,{},{}


{'next_page_token': ''}

,model,version,run_id,tags
0,planilha-turmas-demo,15,c0d9f3138c874049b339e0f54ab8348c,{'execution_id': '712b11cd-a96a-4bb9-83bb-6917...
1,planilha-turmas-demo,14,7090206ddf77414892faadd9047bb040,{'execution_id': '2f6e1b6d-07f8-40a7-812d-df1b...
2,planilha-turmas-demo,13,4aac59f96a4b448f918be9e01810dba8,{'execution_id': '81d0a4af-b0d4-4a3f-b99e-6711...
3,planilha-turmas-demo,12,e5eb660aa44b4542b7281105c34613da,{'execution_id': '61541e92-a80a-4041-8592-c8b8...
4,planilha-turmas-demo,11,6d3153d23589442b8475e969f82a1d98,{'execution_id': '8d373b5a-f73d-4daa-b5cb-cde6...
5,planilha-turmas-demo,10,69fdee4b8f744f88b03117dfa2eaa5bb,{'execution_id': '93519267-7809-4e64-9e04-a79b...
6,planilha-turmas-demo,9,3d8f1eaeaa79412e90074567e6e2319b,{'execution_id': '46be6191-4d5c-4124-a65a-b8f1...
7,planilha-turmas-demo,8,03012b122e6143b2919807cd91973451,{'execution_id': 'd6260507-b9f4-4011-a3f6-b3dd...
8,planilha-turmas-demo,7,d16bb48748694adc95190ce52b2361ac,{'execution_id': '496e6a0c-3128-4f26-bfe1-b3f6...
9,planilha-turmas-demo,6,147fd510dc1044ebb6b97476ee4c0fc3,{'execution_id': '22d9ba74-2344-4995-9766-d379...


""


2. Inspecionar artefatos e proveniência da versão selecionada. Esperado: hashes iguais; nenhuma aprovação implícita.

In [3]:
if params.model_version:
    from src.operations.training import trace_for
    from src.workbench.verification import verify_model_storage
    trace=trace_for(client,client.get_model_version(params.model_name,params.model_version))
    evidence["candidate"]=trace;evidence["storage"]=verify_model_storage(trace)
    display(trace);display(pd.DataFrame(evidence["storage"]))


3. Ação opt-in: solicitar revisão ou decidir pedido identificado. APPROVE/REVOKE/ROLLBACK requerem justificativa; reviewer vem da credencial, não do texto digitado.

In [4]:
if params.governance_action!="read":
    params.authorize()
    if params.governance_action=="request":
        if not params.model_version or len(params.reason)<8: raise ValueError("Versão e justificativa obrigatórias")
        evidence["review"]=call(cfg,"POST","/v1/governance/requests",dict(model_name=params.model_name,model_version=params.model_version,
            action=params.review_action,reason=params.reason,requested_by=params.requester))
    else:
        if not params.approval_id: raise ValueError("approval_id obrigatório")
        evidence["review"]=call(cfg,"POST",f"/v1/governance/requests/{params.approval_id}/review",
            dict(approve=params.governance_action=="approve",confirm="REVIEW_EXPLICIT_VERSION"),operator=True)
    display(evidence["review"])
else:
    evidence["review"]="NOT_EXECUTED: leitura"


Guardar evidência desta execução e links. Esperado: caminho do relatório, sem credenciais.

In [5]:
report_status="PARTIAL" if any(isinstance(value,str) and value.startswith("NOT_EXECUTED") for value in evidence.values()) else "PASS"
display(report(NOTEBOOK_ID,evidence,status=report_status,limitations=["Dados DEMO e contrato provisório; regras não avaliadas não são homologação", "Execução sem erro do notebook não implica execução de operações opt-in desativadas"]))


'/reports/v2/07_registry_governanca.json'